# Module 03: Styling and Annotation

We now have a range of plot types available: line plots, scatter plots, bar charts, histograms, and more. This module turns attention to how a plot looks. That might sound like a cosmetic concern, but it is not: the clarity of axis labels, the readability of tick marks, and the visual distinction between data series are all part of how scientific information gets communicated. A well-styled figure requires no explanation beyond itself.

## Imports and Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

## Shared data: molecular weight distributions

Most examples in this module use synthetic molecular weight distribution (MWD) curves for two hypothetical polymers. Each distribution is a bimodal blend of two Gaussians on a log(M) axis, reflecting the kind of profile you would see from an acrylate polymer produced by free-radical polymerization at high conversion, where significant chain transfer to polymer and subsequent terminal double-bond polymerization lead to the formation of a highly branched 'gel' fraction alongside the linear 'sol' population. We work directly in log space here, treating log₁₀(M) as the x-axis variable. Plotting on a true logarithmic scale (so the tick labels read in g/mol rather than log units) is covered in Module 04.

In [ ]:
# x axis: log10 of molecular weight, ranging from ~10^3 to ~10^9 g/mol
log_mw = np.linspace(3, 9, 500)

def bimodal_mwd(log_mw, mu1, sigma1, w1, mu2, sigma2, w2):
    """Weighted sum of two Gaussians in log(M) space."""
    g1 = w1 * np.exp(-((log_mw - mu1) ** 2) / (2 * sigma1 ** 2))
    g2 = w2 * np.exp(-((log_mw - mu2) ** 2) / (2 * sigma2 ** 2))
    return g1 + g2

# Polymer A: 30% low-MW fraction, 70% high-MW fraction
mwd_a = bimodal_mwd(log_mw,
                    mu1=4.5, sigma1=0.40, w1=0.30,
                    mu2=7.3, sigma2=0.15, w2=0.70)

# Polymer B: 60% low-MW fraction, 40% high-MW fraction (narrower peaks)
mwd_b = bimodal_mwd(log_mw,
                    mu1=4.8, sigma1=0.35, w1=0.60,
                    mu2=7.6, sigma2=0.18, w2=0.40)

# Normalize so both curves integrate to the same area (for a fair visual comparison)
mwd_a = mwd_a / np.trapezoid(mwd_a, log_mw)
mwd_b = mwd_b / np.trapezoid(mwd_b, log_mw)

Normalizing by the integral under each curve means the y-axis represents a probability density in log(M) space, so both distributions are visually comparable regardless of the total mass injected. This is the standard representation for GPC/SEC data.

## Axis labels and title

Axis labels are the minimum required annotation for any figure that will be read by someone other than you. They need to include the quantity name and the units. The `fontsize` argument controls the text size on any `ax.set_*` call, and `fontfamily` (or `fontname`) sets the typeface. Matplotlib ships with a set of generic font families: `'sans-serif'`, `'serif'`, and `'monospace'`, as well as specific named fonts if they are installed on the system.

In [ ]:
fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a, color='#2166ac')

ax.set_xlabel('log₁₀(M) (g/mol)', fontsize=13, fontfamily='sans-serif')
ax.set_ylabel('Normalized mass', fontsize=13, fontfamily='sans-serif')
ax.set_title('Molecular weight distribution', fontsize=15, fontweight='bold')
plt.show()

Units belong in parentheses at the end of the label: `'log₁₀(M) (g/mol)'`. This is the convention in most journals and is immediately understood by any scientific reader. If the quantity is dimensionless, you can either omit units entirely or write `'(dimensionless)'`.

## Colors

Matplotlib accepts colors in several formats: named colors (e.g., `'steelblue'`), hex codes (e.g., `'#2c7bb6'`), and RGB tuples with values between 0 and 1 (e.g., `(0.17, 0.48, 0.71)`). Below you see a list of color names. Named colors are convenient but the selection is limited and not always visually balanced across a series. For multi-series scientific figures, a carefully chosen discrete palette is more reliable. More on this in Module 13.

![Color Palette](https://raw.githubusercontent.com/kiarashfa/pyscinote/main/visualization/assets/Colors.png)

Here we have used named colors, but the hex codes are also easy to copy into any figure. Once you have a palette you like, it is worth defining it at the top of your notebook and referencing it by index throughout, which keeps all your figures visually consistent.

In [ ]:
fig, ax = plt.subplots()

# Simply add color by writing the name
ax.plot(log_mw, mwd_a, color='steelblue', label='Polymer A')
ax.plot(log_mw, mwd_b, color='darkorange', label='Polymer B')

# Or use the hex color code (Try after by uncommenting this)
# ax.plot(log_mw, mwd_a, color='#4682B4', label='Polymer A')
# ax.plot(log_mw, mwd_b, color='#FF8C00', label='Polymer B')

ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')
plt.show()

## Legends

A legend is needed whenever you have more than one data series on a plot. You add a label to each `ax.plot()` call and then call `ax.legend()` to render it. Matplotlib can place the legend automatically or you can specify the location explicitly.

In [ ]:
fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a, color='steelblue', label='Polymer A')
ax.plot(log_mw, mwd_b, color='darkorange', label='Polymer B')
ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')

ax.legend(
    loc='upper right',   # explicit placement
    frameon=False,       # no border box (or use fancybox)
    fontsize=11,         # control the size
    ncol=1               # one column; set to 2 for side-by-side entries
)

plt.show()

The `loc` argument accepts strings like `'upper right'`, `'lower left'`, `'center'`, and `'best'`. The `'best'` option asks Matplotlib to find the position that overlaps the data least, which works well in simple cases. For publication figures, explicit placement is more reliable. Removing the frame with `frameon=False` gives a cleaner result in most contexts.

## Grid lines

Grid lines help a reader trace a value from a data point back to the axis. Matplotlib draws both major grids (aligned with the labeled tick marks) and minor grids (at subdivisions between them). Both are off by default.

In [ ]:
fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a, color='steelblue', label='Polymer A')
ax.plot(log_mw, mwd_b, color='darkorange', label='Polymer B')

# Major grid: aligned with labeled ticks
ax.grid(which='major', linestyle='--', linewidth=0.6, color='grey', alpha=0.6)

# Minor grid: requires minor ticks to be enabled first
ax.minorticks_on()
ax.grid(which='minor', linestyle=':', linewidth=0.4, color='grey', alpha=0.3)

ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')
ax.legend(frameon=False, fontsize=10)
plt.show()

Keep grid lines visually subordinate to the data. Low `alpha` values (0.3 to 0.6) and thin lines (0.4 to 0.8 pt) achieve this. A grid that competes with the data for attention is worse than no grid at all.

## Tick marks

Matplotlib sets tick positions and labels automatically, but you often need more control: a different tick interval, larger tick labels for a figure that will be reduced in a journal layout, rotated labels to avoid overlap, or a specific number format. All of this goes through the `ax.xaxis` and `ax.yaxis` objects, or through the `ticker` module imported above.

In [ ]:
fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a, color='steelblue', label='Polymer A')
ax.plot(log_mw, mwd_b, color='darkorange', label='Polymer B')

# Place major ticks at every integer log unit (3, 4, 5, 6, 7, 8)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

# Format y tick labels to two decimal places
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.25))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

# Increase tick label font size
ax.tick_params(axis='both', labelsize=11)

# Rotate x tick labels to avoid crowding
ax.tick_params(axis='x', rotation=45)

ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')
ax.legend(frameon=False, fontsize=10)
plt.show()

`ticker.MultipleLocator(base)` places ticks at every multiple of `base`. `ticker.FormatStrFormatter` uses a C-style format string: `'%.2f'` gives two decimal places, `'%.2e'` gives scientific notation, and `'%d'` gives integers. These two locators cover the majority of tick-formatting needs in scientific figures.

## Scientific notation on axes

When axis values span several orders of magnitude, or when the numbers are very large or very small, scientific notation keeps labels from becoming unreadably wide. Matplotlib handles this through `ScalarFormatter` and the `ticklabel_format` convenience method.

In [ ]:
# Simulate detector response on MWD
detector_counts = mwd_a * 1.5e6

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left panel: default formatting (often awkward with large numbers)
axes[0].plot(log_mw, detector_counts)
axes[0].ticklabel_format(axis='y', style='plain')
axes[0].set_title('Default tick labels')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Detector response (counts)')

# Right panel: scientific notation forced on y axis
axes[1].plot(log_mw, detector_counts)
axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[1].set_title('Scientific notation on y axis')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Detector response (counts)')

plt.tight_layout()
plt.show()

`scilimits=(0, 0)` forces scientific notation regardless of the magnitude. The multiplier (e.g., ×10⁶) appears as an offset label at the top of the axis. If you want scientific notation only outside a particular range, you can set `scilimits=(-3, 4)`, which means: use scientific notation when the exponent is less than -3 or greater than 4.

## Marker symbols and line styles

When plotting discrete data points, or when you need to distinguish series in a figure that will be printed in greyscale, different marker shapes are as useful as different colors. Many journals still print in black and white, and reviewers reading a PDF on a greyscale display will lose color as a distinguishing cue entirely. Combining a distinct marker with each series means the figure remains readable under any reproduction conditions. Matplotlib also has four built-in line styles, controlled by the `linestyle` argument. Like marker shapes, line styles are useful for distinguishing series in greyscale or for readers with color vision deficiencies.

The cell below is a reference chart: each marker is plotted with its string code so you can copy the one you need.

In [ ]:
markers = [
    ('o', 'circle'),
    ('s', 'square'),
    ('^', 'triangle up'),
    ('v', 'triangle down'),
    ('D', 'diamond'),
    ('P', 'plus (filled)'),
    ('X', 'x (filled)'),
    ('*', 'star'),
    ('h', 'hexagon'),
    ('p', 'pentagon'),
]

fig, ax = plt.subplots(figsize=(10, 2.5))

for i, (marker_code, marker_name) in enumerate(markers):
    ax.plot(i, 1, marker=marker_code, markersize=14,
            color='#2166ac', linestyle='none')
    ax.text(i, 0.55, f"'{marker_code}'", ha='center', fontsize=9,  color='#444444')
    ax.text(i, 0.3,  marker_name,        ha='center', fontsize=7.5, color='#888888')

ax.set_xlim(-0.8, len(markers) - 0.2)
ax.set_ylim(0, 1.5)
ax.axis('off')
ax.set_title('Marker reference', fontsize=12, pad=10)

plt.show()

Now here is the same MWD comparison plotted with sampled points and distinct markers instead of continuous lines, as you would present it if your GPC software exported discrete retention-time points rather than a smoothed curve.

In [ ]:
# Sample every 25th point so the markers are spaced and readable with markevery
linestyles = [('-',  'solid'), ('--', 'dashed'), (':',  'dotted'), ('-.', 'dash-dot'),]
fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a,
        marker='o', markersize=5, markevery=25,
        linestyle='-', linewidth=1.5,
        color='#4682B4', label='Polymer A')

ax.plot(log_mw, mwd_b,
        marker='s', markersize=5, markevery=25,
        linestyle='-', linewidth=1.5,
        color='#FF8C00', label='Polymer B')

ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')
ax.set_title('MWD with markers (readable in greyscale)')
ax.legend(frameon=False, fontsize=11)

plt.show()

All marker codes are passed as the `marker` argument to `ax.plot()` or `ax.scatter()`. Marker size is controlled by `markersize` in `ax.plot()` and by `s` (in points squared) in `ax.scatter()`. Solid lines read most clearly at any line weight. Dashed lines work well for a second series or for a fitted curve overlaid on data. Dotted and dash-dot lines are best reserved for reference lines or theoretical predictions, where you want them visually distinct but secondary.

## Filling under a curve

`ax.fill_between()` shades the area between two curves, or between a curve and a horizontal reference level. For an MWD, a natural use is to distinguish the sol fraction from the gel fraction. Chains above approximately 10⁷ g/mol (log M = 7) are typically considered gel-forming in network polymers; the fill highlights how much of the distribution falls in each regime.

In [ ]:
gel_threshold = 7.0  # log10(M) = 7 corresponds to M = 10^7 g/mol

fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a, color='#2166ac', linewidth=2, label='Polymer A')

# Sol fraction: below the gel threshold
ax.fill_between(log_mw, mwd_a, 0,
                where=(log_mw < gel_threshold),
                color='#4dac26',
                alpha=0.25,
                label='Sol fraction (M < 10⁷)')

# Gel fraction: above the gel threshold
ax.fill_between(log_mw, mwd_a, 0,
                where=(log_mw >= gel_threshold),
                color='#d6604d',
                alpha=0.25,
                label='Gel fraction (M ≥ 10⁷)')

# Vertical reference line at the gel threshold
ax.axvline(gel_threshold, color='black', linewidth=0.9, linestyle='--')

ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')
ax.set_title('Sol and gel fractions')
ax.legend(frameon=False, fontsize=10)

plt.show()

The `where` argument takes a boolean array that selects which x positions to fill. The `alpha` argument controls transparency: 0.0 is invisible, 1.0 is fully opaque. Values around 0.2 to 0.3 keep the fill visible without obscuring the curve or any grid lines behind it.

For figures destined for print, or for journals that discourage color, hatching patterns provide a greyscale-safe alternative. The example below hatches the region where both MWDs overlap, which is the zone where the two polymers are most directly in competition.

In [ ]:
# The overlap at each point is the minimum of the two distributions
mwd_overlap = np.minimum(mwd_a, mwd_b)

fig, ax = plt.subplots()

ax.plot(log_mw, mwd_a, color='#2166ac', linewidth=2, label='Polymer A')
ax.plot(log_mw, mwd_b, color='#d6604d', linewidth=2, label='Polymer B')

# Lightly fill both distributions
ax.fill_between(log_mw, mwd_a, 0, color='#2166ac', alpha=0.10)
ax.fill_between(log_mw, mwd_b, 0, color='#d6604d', alpha=0.10)

# Hatch the overlapping region
ax.fill_between(log_mw, mwd_overlap, 0,
                facecolor='none',
                edgecolor='#444444',
                hatch='////',
                linewidth=0.0,
                label='Overlap region')

ax.set_xlabel('log₁₀(M) (g/mol)')
ax.set_ylabel('Normalized mass')
ax.set_title('Overlap between two MWDs (hatched)')
ax.legend(frameon=False, fontsize=10)

plt.show()

Common hatch strings include `'/'`, `'\\'`, `'|'`, `'-'`, `'+'`, `'x'`, `'o'`, and `'.'`. Repeating the character (e.g., `'////'`) increases the density of the pattern. Setting `facecolor='none'` and relying only on `edgecolor` and `hatch` produces a clean result that does not depend on color to convey meaning.